- Эта тетрадь предназначена для вычисления gram_structure и косинусной близости

**Вопросы:** 
1. Стоит ли вычислять грамматическую структуру отдельно или заведомо хранить ее в бд.;
2. Вычислять грамматическую структуру предложения или концепта (что лучше);
3. Вычислять косинусную близость предложения или концепта.

### Константы

In [11]:
# импорты
import pandas as pd
from openpyxl import load_workbook
from tqdm import tqdm
tqdm.pandas()


# файл
# FILENAME = '../results/Разметка_сравнение_RU_EN.xlsx'
FILENAME = '../results/примеры.xlsx'

# 
THRESHOLD_EQUIV   = 0.90   # ≥0.90 → equivalent по умолчанию
THRESHOLD_SHIFT   = 0.75   # 0.75–0.89 → shift

### Этап I - обработка и вычленение gram_structure

#### Грамматическая структура: подготовка

In [ ]:
import sys
sys.path.insert(0, '../src')
from grammar import parse_ru, parse_en

#### функции

In [ ]:
tests = [
    ("RU", "запах гари"),
    ("RU", "невыносимый смрад"),
    ("RU", "грузовик дохнул раскаленной вонью"),
    ("EN", "smell of burning"),
    ("EN", "unbearable stench"),
    ("EN", "truck breathed scorching stench"),
]
print("=" * 65)
for lang, phrase in tests:
    fn = parse_ru if lang == "RU" else parse_en
    print(f"\n[{lang}] «{phrase}»")
    print(f"  → {fn(phrase)}")

#### Обработка и вычленение gram_structure

In [ ]:
df = pd.read_excel(FILENAME, sheet_name='Разметка', header=0)
print("Доступные колонки:", list(df.columns))

df['gram_structure_RU'] = df['concept_unit_RU'].apply(
    lambda x: parse_ru(str(x)) if pd.notna(x) else ''
)
df['gram_structure_EN'] = df['concept_unit_EN'].apply(
    lambda x: parse_en(str(x)) if pd.notna(x) else ''
)
print("✅ Обработка завершена")

#### Сохранение

In [19]:
# 2. Открываем оригинал через openpyxl
wb = load_workbook(FILENAME)
ws = wb['Разметка']

# 3. Находим колонки (один раз)
gram_col = None
gram_en_col = None
for col in range(1, ws.max_column + 1):
    val = ws.cell(2, col).value
    if val == 'gram_structure_RU':
        gram_col = col
    elif val == 'gram_structure_EN':
        gram_en_col = col

In [20]:
# 4. Записываем результаты (построчно, но быстро)
for idx, (_, row) in enumerate(df.iterrows()):
    excel_row = idx + 3
    if gram_col:
        ws.cell(excel_row, gram_col).value = row['gram_structure_RU']
    if gram_en_col:
        ws.cell(excel_row, gram_en_col).value = row['gram_structure_EN']

# 5. Сохраняем
wb.save(FILENAME)
print(f"✅ Готово! Обновлено {len(df)} строк")

✅ Готово! Обновлено 1014 строк


### Этап II - Косинусная близость

#### Косинусная близость

In [21]:
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_excel(FILENAME, sheet_name='Разметка', header=0)

In [23]:
model = SentenceTransformer('sentence-transformers/LaBSE')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2651.92it/s]


In [24]:
# ru_embeddings = model.encode(
#     df['concept_unit_RU'].fillna('').astype(str).tolist(),
#     batch_size=32,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

# print("Кодирование английских фраз...")
# en_embeddings = model.encode(
#     df['concept_unit_EN'].fillna('').astype(str).tolist(),
#     batch_size=32,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

In [25]:
ru_embeddings = model.encode(
    df['sent_text_RU'].fillna('').astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Кодирование английских фраз...")
en_embeddings = model.encode(
    df['sent_text_EN'].fillna('').astype(str).tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 32/32 [01:28<00:00,  2.76s/it]


Кодирование английских фраз...


Batches: 100%|██████████| 32/32 [01:11<00:00,  2.24s/it]


In [26]:
# Вычисляем косиносное сходство
df['cosine_sim_LaBSE'] = (ru_embeddings * en_embeddings).sum(axis=1)
df


,case_iD,sent_ID,sent_text_RU,token_ID,token_RU,token_pos,concept_unit_RU,type_RU,connotation_RU,тональность_RU,...,token_pos_EN,gram.structure_EN,type_EN,connotation_EN,тональность_EN,comments_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified
0,NaN,NaN,За все годы лихорадочной работы в моргах ― с ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.849738,NaN,NaN
1,NaN,NaN,"Когда река успокоилась, кто-то переплыл на ос...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.878006,NaN,NaN
2,NaN,NaN,В палатах стоял тяжелый смрад – лежачие стару...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.833219,NaN,NaN
3,NaN,NaN,"Там, под троллейбусом, была тьма и жуткая тес...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.695375,NaN,NaN
4,NaN,NaN,Раздался легкий выстрел — и смрад на всю дере...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.684042,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,NaN,NaN,"На следующее утро, когда Гарри вышел к завтра...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.883550,NaN,NaN
1010,NaN,NaN,"Вошли Дудли с дядей Верноном, брезгливо морща...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.828051,NaN,NaN
1011,NaN,NaN,Вскоре хижина наполнилась запахом потрескивав...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.740455,NaN,NaN
1012,NaN,NaN,"Потом они посетили аптеку, где было достаточн...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.759488,NaN,NaN


#### Разметка translation_shift и shift_notes ##

In [27]:
def classify_shift(row):
    cos = row['cosine_sim_LaBSE']
    
    if cos >= THRESHOLD_EQUIV:
        return 'equivalent'
    elif cos >= THRESHOLD_SHIFT:
        return 'shift'
    else:
        return 'significant_shift'

df['translation_shift'] = df.apply(classify_shift, axis=1)
df 

,case_iD,sent_ID,sent_text_RU,token_ID,token_RU,token_pos,concept_unit_RU,type_RU,connotation_RU,тональность_RU,...,token_pos_EN,gram.structure_EN,type_EN,connotation_EN,тональность_EN,comments_EN,translation_shift,cosine_sim_LaBSE,shift_notes,verified
0,NaN,NaN,За все годы лихорадочной работы в моргах ― с ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.849738,NaN,NaN
1,NaN,NaN,"Когда река успокоилась, кто-то переплыл на ос...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.878006,NaN,NaN
2,NaN,NaN,В палатах стоял тяжелый смрад – лежачие стару...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.833219,NaN,NaN
3,NaN,NaN,"Там, под троллейбусом, была тьма и жуткая тес...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.695375,NaN,NaN
4,NaN,NaN,Раздался легкий выстрел — и смрад на всю дере...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.684042,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1009,NaN,NaN,"На следующее утро, когда Гарри вышел к завтра...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.883550,NaN,NaN
1010,NaN,NaN,"Вошли Дудли с дядей Верноном, брезгливо морща...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.828051,NaN,NaN
1011,NaN,NaN,Вскоре хижина наполнилась запахом потрескивав...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,significant_shift,0.740455,NaN,NaN
1012,NaN,NaN,"Потом они посетили аптеку, где было достаточн...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,shift,0.759488,NaN,NaN


#### Сохранение

In [ ]:
#  Сохраняем результаты в отдельный файл (если надо)
df.to_excel('../results/tables/result_with_shifts.xlsx', index=False)
print("✅ Готово!")


✅ Готово!
